In [5]:
#Библиотеки
from math import ceil, sqrt
from pathlib import Path # Библиотека для работы с файлами и их адресами
from scipy.stats import f
from scipy.stats import chi2
from scipy.stats import norm
import math # Библиотка для математический действий
import pandas as pd # Красивые таблицы в питоне
from random import randint



def calculate_n (d):# Высчитываем объем группы для одной выборки
   y = 0.95 # надежность из условия из условия
   b = 3 # точность оценки математического ожидания из условия
   f_t = (y+1) / 2
   t = norm.ppf(f_t)
   n = t**2 * d**2 / b**2
   n = int(math.ceil(n))  # округляем в большую сторону чтобы n было целым числом
   return n

def lets_make_a_groops (numbers, n, N): # создаем 1 случайную выборку с объемом n
    all_selections = []
    for num in range(n):
        i = randint(0, N - 1) # выбор случайных чисел по условию репрезитативности выборки
        all_selections.append(numbers[i]) # добавление элемента выборку
    return all_selections # возвращает выборку

def calculating_dispersia_s (selection, n, k): # функция для расчета  ско исправленной дисперсии
    sn = set(selection) # уникальные значения выбокри
    x_midl = sum([i * selection.count(i) for i in sn]) / n # среднее
    d = sum([selection.count(i) * (i - x_midl)**2 for i in sn]) / k # исправленная дисперсия
    return d

def cacl_observed_f (s1, D, k):# функция для расчета  F набл.
    x_nabl = k * s1/ D
    return x_nabl

a = 0.05 # Уровень значимости

# Вывод
def output (criet_x_right, criet_x_left, criet_x_two_sided1, criet_x_two_sided2, observed_x):
    print(f"\n--- а) H1: D > σ² ---")
    print(f"X_кр (правосторонняя, α={a}) = {criet_x_right:.4f}")
    if observed_x < criet_x_right:
        print("Вывод: нет оснований отвергнуть H0.")
    else:
        print("Вывод: H0 отвергается.")
    print(f"\n--- б) H1: D != σ² ---")
    print(f"X_кр1 (левая, α/2={a/2}) = {criet_x_two_sided1:.4f}")
    print(f"X_кр2 (правая, α/2={a/2}) = {criet_x_two_sided2:.4f}")
    if criet_x_two_sided1 < observed_x < criet_x_two_sided2:
        print("Вывод: нет оснований отвергнуть H0.")
    else:
        print("Вывод: H0 отвергается.")
    print(f"\n--- в) H1: D < σ² ---")
    print(f"X_кр (левосторонняя, α={a}) = {criet_x_left:.4f}")
    if  observed_x > criet_x_left:
        print("Вывод: нет оснований отвергнуть H0.")
    else:
        print("Вывод: H0 отвергается.")

#Вычисление критических критериев
def criet_x (observed_x, k):
    criet_x_right = chi2.ppf(1 - a, k) # вычисления x критического для правостороннего
    criet_x_left = chi2.ppf(a, k)# вычисления x критического для левостороннего

    #Вычисления x критических для двухсторонней
    criet_x_two_sided1 = chi2.ppf(a/2, k)
    criet_x_two_sided2 = chi2.ppf(1 - a/2, k)

    # Вывод
    output(criet_x_right, criet_x_left, criet_x_two_sided1, criet_x_two_sided2, observed_x)


def task (selection1, D):
    # Объем
    n = len(selection1)
    # Степень свободы
    k = n - 1
    # Исправленная дисперсия:
    s = calculating_dispersia_s (selection1, n, k)
    # Наблюдаемое
    observed_x = cacl_observed_f(s, D, k)
    print(f"---- Гипотеза о генеральной дисперсии ----")
    print(f"n = {n}   k = {k}")
    print(f"D = {D:.4f}")
    print(f"s^2 = {s:.4f}")
    print(f"X набл = {observed_x:.4f}")
    criet_x(observed_x, k)



fp = Path("./Москва_2021.txt")
if fp.exists():
    with open(fp, "r", encoding="utf-8") as file: # Открыть и прочитать файл, на любом языке с переменной файла file
        numbers = sorted([int(line.strip()) for line in file if line.strip()])
    sn = sorted(set(numbers))
    N = 0 # Сумма частот
    x_midl = 0 # Среднее +
    for i in sn:
        n = numbers.count(i) # кол. элемента
        x_midl += i * n # высчитываем сумму среднего, но пока без деления на сумму частот
        N+= n
    x_midl /= N
    D = 0
    for i in sn:
        D += (i - x_midl)**2 * numbers.count(i)
    D = D / N # Дисперсия
    d = math.sqrt(D) # среднее квадратичное отклонение
    n = calculate_n(d) # высчитываем объем
    all_selections = lets_make_a_groops(numbers, n, N) # получаем список списков групп + список средних
    task(all_selections, D) # строим интервальный ряд
else:
    print("Скачайте или создайте файл Москва_2021.txt")

---- Гипотеза о генеральной дисперсии ----
n = 62   k = 61
D = 144.9170
s^2 = 142.3533
X набл = 59.9209

--- а) H1: D > σ² ---
X_кр (правосторонняя, α=0.05) = 80.2321
Вывод: нет оснований отвергнуть H0.

--- б) H1: D != σ² ---
X_кр1 (левая, α/2=0.025) = 41.3031
X_кр2 (правая, α/2=0.025) = 84.4764
Вывод: нет оснований отвергнуть H0.

--- в) H1: D < σ² ---
X_кр (левосторонняя, α=0.05) = 44.0379
Вывод: нет оснований отвергнуть H0.
